# Deep Learning 048 — Backpropagation in a CNN, Part 2

The three operations an ANN does not have. **Flatten and pooling have no
parameters, so you do not differentiate them — you reverse them.** Flatten
backward is a reshape; max pooling backward gives the gradient to **only the
element that won**, and exactly zero to the rest.

| Part | What we check |
|---|---|
| A | flatten backward is a reshape |
| B | max pooling backward: one winner per window, three zeros |
| C | convolution backward, and the full gradient check |
| D | max against average pooling backward |

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# 6x6 image -> conv 3x3 -> ReLU -> maxpool 2x2 -> flatten(4) -> dense -> sigmoid -> BCE
X = rng.normal(size=(6, 6))
y = 1.0
W1 = rng.normal(size=(3, 3)) * 0.5     # 9 parameters
b1 = 0.1                               # 1
W2 = rng.normal(size=4) * 0.5          # 4
b2 = -0.2                              # 1

def forward(X, W1, b1, W2, b2):
    Z1 = np.zeros((4, 4))
    for i in range(4):
        for j in range(4):
            Z1[i, j] = np.sum(X[i:i+3, j:j+3] * W1) + b1
    A1 = np.maximum(0, Z1)                       # ReLU
    P = np.zeros((2, 2)); idx = {}
    for i in range(2):
        for j in range(2):
            win = A1[i*2:i*2+2, j*2:j*2+2]
            P[i, j] = win.max()
            a, b = np.unravel_index(win.argmax(), win.shape)
            idx[(i, j)] = (i*2 + a, j*2 + b)     # remember which element won
    flat = P.reshape(-1)
    Z2 = float(np.dot(flat, W2) + b2)
    A2 = 1 / (1 + np.exp(-Z2))
    loss = -(y * np.log(A2) + (1 - y) * np.log(1 - A2))
    return loss, dict(Z1=Z1, A1=A1, P=P, idx=idx, flat=flat, Z2=Z2, A2=A2)

## Part A — Flatten backward

In [ ]:
loss, c = forward(X, W1, b1, W2, b2)
dZ2 = c["A2"] - y
dflat = dZ2 * W2                      # 4 values, one per pooled position
dP = dflat.reshape(2, 2)              # that is the whole backward pass

print("gradient at the flatten:", np.round(dflat, 5))
print("reshaped to the pooled map:\n", np.round(dP, 5))
assert dP.shape == (2, 2) and np.allclose(dP.reshape(-1), dflat)
print("\nNo derivative involved. Flatten rearranged numbers; its backward pass")
print("rearranges them back.")

## Part B — Max pooling backward

In [ ]:
dA1 = np.zeros((4, 4))
for (i, j), (a, b) in c["idx"].items():
    dA1[a, b] = dP[i, j]              # the winner takes all

print("gradient reaching the ReLU output:\n", np.round(dA1, 5))
zeros = int((dA1 == 0).sum())
print(f"\n{zeros} of {dA1.size} positions got exactly zero "
      f"({zeros/dA1.size:.0%}) - one winner per 2x2 window")
assert zeros == 12
print("\nThat is why the forward pass has to remember which element won:")
print("a small, real memory cost for every pooling layer.")

## Part C — Convolution backward, and the check

In [ ]:
dZ1 = dA1 * (c["Z1"] > 0)             # ReLU: pass where the input was positive

# dW1[u,v] = sum over output positions of dZ1 * the input pixel that met it
dW1 = np.zeros((3, 3))
for u in range(3):
    for v in range(3):
        dW1[u, v] = np.sum(dZ1 * X[u:u+4, v:v+4])
db1 = dZ1.sum()

def numerical(setter, value, eps=1e-6):
    return (setter(value + eps) - setter(value - eps)) / (2 * eps)

print(f"{'':>8}{'analytic':>14}{'numerical':>14}")
ok = True
for u in range(3):
    for v in range(3):
        def setter(val, u=u, v=v):
            W1c = W1.copy(); W1c[u, v] = val
            return forward(X, W1c, b1, W2, b2)[0]
        num = numerical(setter, W1[u, v])
        ok &= bool(np.isclose(dW1[u, v], num, atol=1e-6))
        print(f"W1[{u},{v}]{dW1[u, v]:>14.8f}{num:>14.8f}")
num_b1 = numerical(lambda val: forward(X, W1, val, W2, b2)[0], b1)
print(f"b1      {db1:>14.8f}{num_b1:>14.8f}")
assert ok and np.isclose(db1, num_b1, atol=1e-6)
print("\nAll 15 gradients agree with finite differences: the backward pass is right.")

## Part D — Average pooling backward

In [ ]:
dA1_avg = np.zeros((4, 4))
for i in range(2):
    for j in range(2):
        dA1_avg[i*2:i*2+2, j*2:j*2+2] = dP[i, j] / 4     # every element contributed

print("max pooling backward (sparse):\n", np.round(dA1, 5))
print("\naverage pooling backward (spread):\n", np.round(dA1_avg, 5))
print(f"\nzeros: max {int((dA1==0).sum())} of 16, average {int((dA1_avg==0).sum())} of 16")
assert (dA1_avg == 0).sum() == 0
print("Max pooling gives sparse gradients; average pooling gives every element 1/k^2.")

## What to take away

- **A layer with no parameters is not differentiated — its backward pass reverses
  its forward pass.**
- **Flatten backward is a reshape.**
- **Max pooling backward routes the gradient to the winner** and gives the other
  three **exactly zero** — 12 of 16 positions here.
- That requires **caching which element won**: a real memory cost.
- **Average pooling spreads 1/k² to every element** instead.
- All fifteen gradients matched finite differences, which is how you know.

## Exercises

1. Make the pooling window 4×4 over the whole map. What fraction of positions
   now receive zero gradient?
2. Replace ReLU with sigmoid in the tiny network. Which gradients change, and does
   the check still pass?
3. Why does `dW1` sum over output positions rather than taking a single product?